### Tools
- Models can request to call tools that perform tasks such as fectching data from a database , searching the web or running code.Tools are a pairing of :
1. A schema , including the name of the tool , a descrption and/or argument definitions(often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os

from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke(" why do parrots talk?"
)
response

TypeError: str expected, not NoneType

In [ ]:
from langchain.tools import tool

@tool # converts a Python function into a LangChain Tool that an LLM agent can discover and invoke.
def get_weather(location:str)->str:
    """Get the weather at a location""" #
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("what's the weather like in Boston?")

for tool_call in response.tool_calls:
    # View the tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

print(response)


Tool: get_weather
Args: {'location': 'Boston'}
content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'pwdgrss3d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 154, 'total_tokens': 248, 'completion_time': 0.148607438, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.0083056, 'prompt_tokens_details': None, 'queue_time': 0.006903164, 'total_time': 0.156913038}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'log

### Tool Execution Loops

In [11]:
# Step 1 : Model generates tool calls
messages = [{"role": "user" , "content": " what's the weather in Boston?"}];
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)



# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print(messages)

# step 3 : Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

print (final_response.text)

[{'role': 'user', 'content': " what's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let me format the tool call correctly.\n', 'tool_calls': [{'id': 'j2gbxv7ee', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 153, 'total_tokens': 245, 'completion_time': 0.144978121, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.007881321, 'prompt_tokens_details': None, 'queue_time': 0.008201479, 'total_time': 0.152859442}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_re